# Phase 3: baseline training + PTQ/QAT ladder (Colab / Kaggle GPU)

Runs `edge_ai_compression.experiments.run_phase3`: trains ResNet-18 (and optionally ViT-S)
on CIFAR-10 for 3 seeds, then runs the quantization ladder sweep. Every result is written to
the experiment DB under `results/` (the only results output). Download it at the end and
merge it into your local DB. Numbers never leave the DB by hand.

* Colab: Runtime -> Change runtime type -> GPU. Kaggle: Accelerator -> GPU, Internet on.
* The sweep **resumes**: if the session dies, re-run the cells. Finished variants are skipped
  (keep `results/` and `models/` on Drive, see below).
* Latency columns measured here are from a shared cloud VM (secondary platform, noisy);
  the fingerprint records the CPU. Primary latency numbers come from the M5 Pro.

In [ ]:
BRANCH = "phase3-ptq"
REPO = "https://github.com/eshaan2418/Edge-AI-Model-Compression-Deployment.git"
!git clone --branch $BRANCH $REPO repo
%cd repo
!pip install -q -e ".[kernels,export]"
!bash scripts/build_kernels.sh

In [ ]:
from edge_ai_compression.inference import kernels

print("ISAs:", kernels.supported_isas())

Optional (Colab): persist `results/` and `models/` on Google Drive so an interrupted run resumes.

In [ ]:
import os

if os.path.isdir("/content"):
    from google.colab import drive

    drive.mount("/content/drive")
    root = "/content/drive/MyDrive/edge_ai_phase3"
    for d in ("results", "models"):
        os.makedirs(f"{root}/{d}", exist_ok=True)
        if not os.path.islink(d):
            !rm -rf {d} && ln -s {root}/{d} {d}

Smoke test: same code path, synthetic data, a few seconds.

In [ ]:
SMOKE = "--smoke --seeds 0 --results /tmp/smoke_results --models /tmp/smoke_models"
!python -m edge_ai_compression.experiments.run_phase3 $SMOKE

ResNet-18 track: 3 training seeds (60 epochs each) + 36-run ladder.

In [ ]:
!python -m edge_ai_compression.experiments.run_phase3 --track resnet18 --device cuda --seeds 0 1 2

ViT-S track (SmoothQuant): 3 seeds x 200 epochs + 21-run ladder. Check outlier stats in the docs first.

In [ ]:
!python -m edge_ai_compression.experiments.run_phase3 --track vit_s --device cuda --seeds 0 1 2

Package the experiment DB for download.

In [ ]:
!zip -qr phase3_results.zip results
print("Download phase3_results.zip, unzip it locally, then run:")
print("  python -m edge_ai_compression.experiment_db.merge --src <unzipped>/results --dst results")